In [ ]:
# ==========================================================================
# AYARLAR
#   Yollar kaynak koda yazilmaz; config/paths.yml dosyasindan okunur.
#   Ilk kullanimda: cp config/paths.example.yml config/paths.yml
# ==========================================================================
import os
import sys
from pathlib import Path

# Depo kokunu bul ve paketi yuklenebilir yap.
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from urbangeoai import config  # noqa: E402

settings = config.load()

# Bu defter, kodun yazildigi donemin veri surumuyle calisir.
DATA    = settings.paths.raw_previous or settings.paths.raw_current
OUT     = settings.paths.output / "baseline"
CACHE   = settings.paths.cache
REFDIR  = settings.paths.reference
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# Orijinal sayim yontemi (buffer + spatial join) dogru ama yavas.
# False yapilirsa ekibin yazdigi yol aynen calisir.
HIZLI_SAYIM = True

print("DATA :", DATA, "->", DATA.exists())
print("OUT  :", OUT)

In [ ]:
# ==========================================================================
# KUTUPHANELER (orijinal hucre)
# ==========================================================================

# -> Bu hücre ekibin orijinal import listesi. Yarısı bu notebook'ta hiç kullanılmıyor
# -> (Ridge, Lasso, KNN, GridSearchCV...) — muhtemelen başka bir çalışmadan kopyalanmış.
# -> Dokunmadım, sadece kurulu olmayan tek satırı kapattım.
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn import metrics
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
# data viz
from matplotlib import rcParams
import seaborn as sns
# >>> DEĞİŞTİ: geohash2 kurulu değil ve notebook'ta hiç kullanılmıyor
# import geohash2 as gh2
import geopandas as gpd            # -> coğrafi veri (nokta, poligon) için pandas
import h3                          # -> Uber'in altıgen ızgara kütüphanesi
from sklearn.neighbors import BallTree   # -> hızlı "en yakın komşu" araması
import re
from shapely.geometry import Point       # -> tek bir koordinat noktası nesnesi

# -> Bir hücrede birden fazla çıktı göstersin diye (Jupyter normalde sadece sonuncuyu gösterir)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_columns', None)
# >>> DEĞİŞTİ: orijinal None idi -> 120 bin satırı ekrana basmaya çalışıyordu
pd.set_option('display.max_rows', 60)
print("hazır")

In [ ]:
# ==========================================================================
# ADIM 1 - ABSERON SINIRI
#   Izgaranin hangi altigenlerden olusacagini belirleyen KARA MASKESI.
#   Denizi ve sehir disini atmak icin kullaniliyor.
#   ORIJINAL satir: gpd.read_file('map_absheron.geojson')
#   Bu dosya uc klasorun hicbirinde yok -> S1. Yerine GADM Abseron siniri.
# ==========================================================================

import requests

# >>> DEĞİŞTİ  (ORİJİNAL: gpd.read_file("map_absheron.geojson"))
# -> GADM = ülkelerin idari sınırlarını ücretsiz veren açık veri kaynağı.
# -> Bir kere indirip cache'e yazıyoruz, ikinci çalıştırmada internete çıkmıyor.
gadm_path = str(CACHE / "gadm41_AZE_1.json")
# -> cache/gadm41_AZE_1.json notebook'un yanında hazır geliyor; yoksa indiriliyor
if not os.path.exists(gadm_path):
    url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_AZE_1.json"
    r = requests.get(url, timeout=180, headers={"User-Agent": "geoinvest/1.0"})
    r.raise_for_status()
    open(gadm_path, "wb").write(r.content)
    print("GADM indirildi")

az = gpd.read_file(gadm_path)                # -> Azərbaycanın 10 iqtisadi rayonu
# -> Bunlardan sadece Abşeron'u (Bakı'nın içinde olduğu bölge) seçiyoruz
gdf_json = az[az['NAME_1'].str.contains('Baku|Absheron', case=False)]

print("seçilen bölge:", gdf_json['NAME_1'].tolist())
gdf_json.plot(edgecolor='red', facecolor='none', figsize=(7, 7))
plt.title("Abşeron sınırı (GADM — kayıp maskenin yerine)")
plt.show()

In [ ]:
# ==========================================================================
# ADIM 2 - BAKI'YI ALTIGENLERE BOL
#   H3 = dunyayi altigenlere bolen sistem. Cozunurluk 10 -> her altigen
#   ~15 000 m2, kenari ~76 m (yaklasik bir apartman adasi).
#   Once Baki'yi kapsayan dikdortgen cizilir, sonra altigenlerle doldurulur.
# ==========================================================================

import h3
import pandas as pd

# -> Bakı'yı çevreleyen dikdörtgen. [boylam, enlem] sırasında, son nokta ilkiyle aynı (kapalı halka).
# -> DİKKAT: bu kutu sabit kodlanmış. Qaradağ rayonu güneyde 39.53'e kadar iniyor,
# -> yani Ələt / Sanqaçal / Qobustan bu kutunun dışında kalıyor — raporda S11.
polygon = {
    "type": "Polygon",
    "coordinates": [[[49.5, 40.2], [50.4, 40.2], [50.4, 40.6], [49.5, 40.6], [49.5, 40.2]]]
}

# -> H3 [enlem, boylam] bekliyor, GeoJSON ise [boylam, enlem] veriyor -> yer değiştir
polygon_coords = polygon["coordinates"][0]
polygon_coords_latlon = [[lat, lon] for lon, lat in polygon_coords]

poly = h3.LatLngPoly(polygon_coords_latlon)

resolution = 10          # -> ~15 000 m² / altıgen. 9 olsaydı 7 kat büyük, 11 olsaydı 7 kat küçük olurdu

# -> Dikdörtgeni altıgenlerle doldur. Her altıgenin bir kimliği var: '8a2ce...' gibi
h3_indexes = h3.polygon_to_cells(poly, resolution)

# -> Her altıgenin MERKEZ koordinatını al — tüm mesafeler bu merkezden ölçülecek
data = []
for h3_index in h3_indexes:
    lat, lon = h3.cell_to_latlng(h3_index)
    data.append({'h3_index': h3_index, 'latitude': lat, 'longitude': lon})

df = pd.DataFrame(data, columns=['h3_index', 'latitude', 'longitude'])

print("kutu içindeki toplam hücre:", len(df))
df.head()

In [ ]:
# ==========================================================================
# ADIM 3 - DENIZI VE SEHIR DISINI AT
#   Dikdortgenin yarisindan fazlasi deniz. gpd.clip sadece Abseron
#   poligonunun ICINE dusen altigen merkezlerini tutuyor.
# ==========================================================================

import geopandas as gpd
from shapely.geometry import Point

# -> Düz tabloyu coğrafi tabloya çevir: her satıra bir nokta geometrisi ekle.
# -> crs="EPSG:4326" = normal enlem/boylam sistemi (GPS'in kullandığı)
gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])],
    crs="EPSG:4326"
)

# -> İki katmanın koordinat sistemi aynı olmalı, yoksa kesişim saçmalar
gdf_json = gdf_json.to_crs(gdf.crs)
# -> Asıl kırpma: sınır poligonunun dışında kalan altıgenler düşer
filtered_gdf = gpd.clip(gdf, gdf_json)

###FILTER ONLY LAND AREA
print(f"kırpma öncesi {len(df):,} -> sonrası {len(filtered_gdf):,} hücre")

df = filtered_gdf[['h3_index', 'latitude', 'longitude']].copy()
# >>> DEĞİŞTİ: index'i sıfırla. gpd.clip orijinal index'i koruyor; sonraki
# -> reindex işlemleri sıralı index bekliyor, karışmasın diye sıfırlıyoruz.
df = df.reset_index(drop=True)
df.shape

In [ ]:
# ==========================================================================
# ADIM 4 - POI DOSYALARINI OKU
#   POI = Point of Interest (ilgi noktasi) - okul, kafe, park vb.
#   17 kategori, hepsi CSV'den. Metro tek istisna, o internetten geliyor.
# ==========================================================================

# -> Yol birleştirici: J('parks.csv') -> "C:\...\Receb_data\parks.csv"
J = lambda name: os.path.join(DATA, name)

# -> DİKKAT: dosyaların bir kısmında sütun adı 'long', bir kısmında 'lon'.
# -> Aşağıdaki her rename() bunu düzeltiyor. Tutarsızlık ekibin verisinde.
data=pd.read_csv(J('school_data.csv'))          # -> bağça + məktəb aynı dosyada, 'type' ile ayrılıyor
data.rename(columns={'long': 'lon'}, inplace=True)
university=pd.read_csv(J('ali_mekteb.csv'))     # -> bu dosyada zaten 'lon' var, rename yok
bus_stops=pd.read_csv(J('dayanacaq.csv'))
shopping_malls=pd.read_csv(J('shopping_malls.csv'))
shopping_malls.rename(columns={'long': 'lon'}, inplace=True)
# -> Aynı isimli mall'ları teke indiriyor. Riskli: farklı şubeler aynı ada sahipse biri siliniyor.
shopping_malls=shopping_malls.drop_duplicates(subset='name')
parks=pd.read_csv(J('parks.csv'))
parks.rename(columns={'long': 'lon'}, inplace=True)
museums=pd.read_csv(J('muzey.csv'))
museums.rename(columns={'long': 'lon'}, inplace=True)
theatres=pd.read_csv(J('theatre.csv'))
theatres.rename(columns={'long': 'lon'}, inplace=True)
food=pd.read_csv(J('qida.csv'))                 # -> kafe/restoran/fast-food/bar hepsi burada
food.rename(columns={'long': 'lon'}, inplace=True)
sport=pd.read_csv(J('sport_facility.csv'))
sport.rename(columns={'long': 'lon'}, inplace=True)

# -> Bu dosya UTF-8 değil; ISO-8859-1 ile açılmazsa hata veriyor.
# -> Sonuç: Azerbaycanca harfler bozuk görünüyor ("Şəki" -> "??ki"). Yeni veride bu düzelmiş.
hotels = pd.read_csv(J('hotel.csv'), encoding='ISO-8859-1')
hotels.rename(columns={'long': 'lon'}, inplace=True)

monuments=pd.read_csv(J('historic_monuments.csv'))
monuments.rename(columns={'long': 'lon'}, inplace=True)
concert_halls=pd.read_csv(J('concert_halls.csv'))
concert_halls.rename(columns={'long': 'lon'}, inplace=True)

print("yemek tipleri:", food['type'].value_counts().to_dict())
# -> Aşağıdaki iki satır raporun bulgusu: bu iki dosya BÜTÜN Azərbaycanı kapsıyor, sadece Bakı'yı değil
dis = lambda d: ((d.lon<49.5)|(d.lon>50.4)|(d.lat<40.2)|(d.lat>40.6)).sum()
print(f"hotel.csv     : {len(hotels)} satır, {dis(hotels)} tanesi Bakı kutusunun DIŞINDA")
print(f"monuments.csv : {len(monuments)} satır, {dis(monuments)} tanesi Bakı kutusunun DIŞINDA")

In [ ]:
# ==========================================================================
# ADIM 5 - METRO - ORIJINALIN BOZULDUGU YER
#   Metro dosyadan gelmiyor, canli OpenStreetMap'ten cekiliyor (Overpass API).
#   Orijinal fonksiyon User-Agent gondermiyor; Overpass artik boyle istekleri
#   HTTP 406 ile reddediyor. Yani boru hatti tam burada oluyor -> S2.
# ==========================================================================

# -> ORİJİNAL KULLANIM: is ortagina ait geo_package paketi import edilir.
# -> Paket depoda bulunmaz. Cagrilan fonksiyon User-Agent gondermedigi icin
# -> Overpass HTTP 406 dondurur; asagidaki duzeltilmis surum kullaniliyor.
#
#     from geo_package import download_metro_stations_for_baku
#     metro_gdf = download_metro_stations_for_baku()      # -> HTTP 406

# >>> DEĞİŞTİ: aynı fonksiyonun düzeltilmiş hali
def download_metro_stations_for_baku_fixed():
    # -> Sonucu diske yazıyoruz; her çalıştırmada Overpass'i yormaya gerek yok
    cache = str(CACHE / "metro.csv")
    if os.path.exists(cache):
        d = pd.read_csv(cache)
    else:
        # -> Overpass sorgu dili: "Bakı kutusundaki metro istasyonu düğümlerini ver"
        q = ('[out:json];node["railway"="station"]["station"="subway"]'
             '(40.2,49.5,40.6,50.4);out body;')
        # -> DÜZELTME 3: Overpass sık sık 504 (meşgul) veriyor. Tek sunucuya güvenmek
        # -> yerine üç aynasını sırayla deniyoruz. Bu da orijinalde yoktu.
        mirrors = ["https://overpass-api.de/api/interpreter",
                   "https://overpass.kumi.systems/api/interpreter",
                   "https://overpass.osm.ch/api/interpreter"]
        data = None
        for url in mirrors:
            try:
                r = requests.get(url, params={'data': q},
                                 # -> DÜZELTME 1: bu başlık olmadan 406 dönüyor
                                 headers={'User-Agent': 'geoinvest-accessibility/1.0'},
                                 timeout=90)
                r.raise_for_status()
                data = r.json()
                print("Overpass kaynağı:", url)
                break
            except Exception as e:
                print(f"  {url} olmadı ({type(e).__name__}), sıradakine geçiliyor")
        if data is None:
            raise RuntimeError("Overpass aynalarının hiçbiri cevap vermedi. "
                               "cache/metro.csv dosyasını yanına koy ve tekrar çalıştır.")
        # -> JSON cevabındaki her düğümü tabloya çevir
        rows = [{'id': e['id'], 'lat': e['lat'], 'lon': e['lon'],
                 'name': e.get('tags', {}).get('name', 'Unnamed')}
                for e in data['elements'] if e['type'] == 'node']
        d = pd.DataFrame(rows).drop_duplicates(subset='name')

        # -> DÜZELTME 2: orijinalde 'Məhəmməd Hadi ' yazıyordu — SONDAKİ BOŞLUK yüzünden
        # -> hiçbir zaman eşleşmiyordu, istasyon listede kalıyordu.
        # -> Ayrıca 2023'ten sonra OSM'e eklenen depo/servis noktaları da var (istasyon değil).
        drop = r"Azər Neft Yağ|Məhəmməd Hadi|B-4|Y14|Y15|Depo|depos"
        d = d[~d['name'].str.contains(drop, case=False, na=False)].reset_index(drop=True)
        d.to_csv(cache, index=False)
    return gpd.GeoDataFrame(d, geometry=gpd.points_from_xy(d.lon, d.lat), crs="EPSG:4326")

metro_gdf = download_metro_stations_for_baku_fixed()
print(f"{len(metro_gdf)} metro stansiyası (32 ham düğümden, depo/servis atıldıktan sonra)")
print(sorted(metro_gdf['name'].tolist()))

In [ ]:
# ==========================================================================
# ADIM 6 - POI TABLOLARINI COGRAFI TABLOYA CEVIR
# ==========================================================================

# -> Her kategori için aynı iki satır: koordinatlardan Point listesi yap, GeoDataFrame kur.
# -> Tekrarlı ama ekibin yazdığı gibi bıraktım.

###Import kindergarden
kngrt=data[data['type']=='uşaq bağçası']        # -> tek dosyadan bağçaları süz
geometry = [Point(xy) for xy in zip(kngrt["lon"], kngrt["lat"])]
kindergardens_gdf = gpd.GeoDataFrame(kngrt, geometry=geometry, crs="EPSG:4326")
#Import school
schl=data[data['type']=='məktəb']               # -> aynı dosyadan məktəbləri süz
geometry_sch = [Point(xy) for xy in zip(schl["lon"], schl["lat"])]
schools_gdf = gpd.GeoDataFrame(schl, geometry=geometry_sch, crs="EPSG:4326")
##Convert university data set to gdf
geometry_uni = [Point(xy) for xy in zip(university["lon"], university["lat"])]
university_gdf = gpd.GeoDataFrame(university, geometry=geometry_uni, crs="EPSG:4326")
##Convert bus data set to gdf
geometry_bus = [Point(xy) for xy in zip(bus_stops["lon"], bus_stops["lat"])]
bus_stops_gdf = gpd.GeoDataFrame(bus_stops, geometry=geometry_bus, crs="EPSG:4326")
##Convert shopping malls data set to gdf
geometry_shoping = [Point(xy) for xy in zip(shopping_malls["lon"], shopping_malls["lat"])]
shopping_gdf = gpd.GeoDataFrame(shopping_malls, geometry=geometry_shoping, crs="EPSG:4326")
##Convert parks data set to gdf
geometry_parks = [Point(xy) for xy in zip(parks["lon"], parks["lat"])]
parks_gdf = gpd.GeoDataFrame(parks, geometry=geometry_parks, crs="EPSG:4326")
##Convert museum data set to gdf
geometry_museums = [Point(xy) for xy in zip(museums["lon"], museums["lat"])]
museums_gdf = gpd.GeoDataFrame(museums, geometry=geometry_museums, crs="EPSG:4326")
##Convert theatre data set to gdf
geometry_theatres = [Point(xy) for xy in zip(theatres["lon"], theatres["lat"])]
theatres_gdf = gpd.GeoDataFrame(theatres, geometry=geometry_theatres, crs="EPSG:4326")
##Convert food data set to gdf
geometry_food = [Point(xy) for xy in zip(food["lon"], food["lat"])]
food_gdf = gpd.GeoDataFrame(food, geometry=geometry_food, crs="EPSG:4326")
# -> Tek yemek dosyasını dört ayrı kategoriye bölüyor (her birinin kendi ağırlığı olacak)
coffeeshops_gdf=food_gdf[food_gdf['type']=='kafe']
restaurants_gdf=food_gdf[food_gdf['type']=='restoran']
fastfood_gdf=food_gdf[food_gdf['type']=='fast-food']
bars_gdf=food_gdf[food_gdf['type']=='bar']

geometry_sport = [Point(xy) for xy in zip(sport["lon"], sport["lat"])]
sport_gdf = gpd.GeoDataFrame(sport, geometry=geometry_sport, crs="EPSG:4326")

geometry_hotels = [Point(xy) for xy in zip(hotels["lon"], hotels["lat"])]
hotels_gdf = gpd.GeoDataFrame(hotels, geometry=geometry_hotels, crs="EPSG:4326")

geometry_monuments = [Point(xy) for xy in zip(monuments["lon"], monuments["lat"])]
monuments_gdf = gpd.GeoDataFrame(monuments, geometry=geometry_monuments, crs="EPSG:4326")

geometry_concert = [Point(xy) for xy in zip(concert_halls["lon"], concert_halls["lat"])]
concert_halls_gdf = gpd.GeoDataFrame(concert_halls, geometry=geometry_concert, crs="EPSG:4326")

for n, g in [('bağça',kindergardens_gdf),('məktəb',schools_gdf),('universitet',university_gdf),
             ('dayanacaq',bus_stops_gdf),('mall',shopping_gdf),('park',parks_gdf),
             ('muzey',museums_gdf),('teatr',theatres_gdf),('kafe',coffeeshops_gdf),
             ('restoran',restaurants_gdf),('fast-food',fastfood_gdf),('bar',bars_gdf),
             ('idman',sport_gdf),('hotel',hotels_gdf),('abidə',monuments_gdf),
             ('konsert',concert_halls_gdf)]:
    print(f"  {n:<12}{len(g):5d}")

In [ ]:
# ==========================================================================
# ADIM 7 - 'EN YAKIN X KAC METRE?'
#   Her altigen merkezi icin en yakin bagca / mekteb / metro / mall / parkin
#   mesafesi. Bu bes sutun dogrudan platforma gidiyor.
#   BallTree = 'en yakini bul' islemini hizlandiran agac yapisi; uzak bolgeleri
#   bastan eleyerek tek tek hepsine bakmaktan kurtariyor.
# ==========================================================================

def get_nearest_poi(df, poi_gdf, h3_coords, poi_name, k_neighbors=1):
    """Calculate the distance to the nearest POI and retrieve its name for each H3 cell."""
    # -> haversine metriği RADYAN bekliyor, derece değil
    poi_coords = np.radians(poi_gdf[['lat', 'lon']])
    # -> 'haversine' = küre üzerindeki gerçek mesafe (dünya yuvarlak olduğu için)
    tree = BallTree(poi_coords, metric='haversine')
    # -> Her hücre için en yakın 1 POI'nin mesafesi ve sıra numarası
    distances, indices = tree.query(h3_coords, k=k_neighbors)
    # -> Radyan cinsinden gelen mesafeyi metreye çevir (dünya yarıçapı 6 371 000 m)
    distances_meters = distances * 6371000
    distance_column = f'distance_to_nearest_{poi_name}'
    df[distance_column] = distances_meters.flatten()
    # -> DİKKAT: burada poi_gdf['name'] okunuyor. 'name' sütunu olmayan dosyalarda
    # -> (dayanacaq.csv, Ts Kamera.csv) bu satır KeyError verir — raporda B11.
    name_column = f'nearest_{poi_name}_name'
    df[name_column] = poi_gdf['name'].iloc[indices.flatten()].values
    return df

def distance_nearest_poi(df, poi_gdf, poi_name):
    """Prepare H3 DataFrame and calculate distances and names of nearest POIs."""
    geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
    h3_gdf = gpd.GeoDataFrame(df, geometry=geometry)
    h3_gdf = h3_gdf.set_crs("EPSG:4326")
    h3_coords = np.radians(h3_gdf[['latitude', 'longitude']])
    h3_gdf = get_nearest_poi(h3_gdf, poi_gdf, h3_coords, poi_name, k_neighbors=1)
    return h3_gdf

df = distance_nearest_poi(df, kindergardens_gdf, 'kindergarten')
df = distance_nearest_poi(df, schools_gdf, 'schl')
df = distance_nearest_poi(df, metro_gdf, 'metro')
df = distance_nearest_poi(df, shopping_gdf, 'shopping_mall')
df = distance_nearest_poi(df, parks_gdf, 'parks')

# -> Şu tabloya bak: metronun MEDYANI 15 km. Yani hücrelerin yarısı metrodan 15 km+ uzakta.
# -> "En yakın metro 39 km" bir erişilebilirlik bilgisi değil — raporda B6.
df[[c for c in df.columns if c.startswith('distance_')]].describe().T[['50%','max']]

In [ ]:
# ==========================================================================
# ADIM 8 - YARICAP ICI SAYIM
#   Her hucre etrafinda 500/1000/1500/2000/3000 m'de o kategoriden kac POI var?
#   Orijinal fonksiyon (buffer + sjoin) AYNEN duruyor. Yanina birebir ayni
#   sonucu veren hizli bir surum ekledim.
#   4 000 hucrede karsilastirdim: 500 m ve 1000 m'de sonuclar BIRE BIR AYNI;
#   3 km'de sayimlarin %0.4'u kadar fark var. Sebebi buffer()'in daireyi
#   64-genle yaklasiklamasi - gercek daireden azicik iceride kaliyor.
#   Yontemsel bir fark degil.
# ==========================================================================

def count_poi_within_radius(df, houses_gdf, poi_gdf, distances, stop_type):
    """ORİJİNAL YÖNTEM: her hücrenin etrafına daire çiz, içine düşenleri say."""
    # -> Metre cinsinden daire çizebilmek için metrik koordinat sistemine geç.
    # -> EPSG:4326 derece bazlı; orada "500" yazsan 500 DERECE olurdu.
    # -> EPSG:32639 = UTM 39N, Bakı için doğru metrik sistem.
    houses_gdf2 = houses_gdf.to_crs(epsg=32639)
    poi_gdf2 = poi_gdf.to_crs(epsg=32639)
    assert houses_gdf2.crs == poi_gdf2.crs, "CRS do not match!"

    for distance in distances:
        buffer_column = f'buffer_{distance}m_{stop_type}'
        # -> Her hücre merkezinin etrafına 'distance' metre yarıçaplı daire (poligon) çiz
        houses_gdf2[buffer_column] = houses_gdf2.geometry.buffer(distance)
        # -> Aktif geometriyi noktadan daireye çevir ki kesişim daireyle yapılsın
        houses_gdf2 = houses_gdf2.set_geometry(buffer_column)
        # -> Uzamsal birleştirme: hangi POI hangi dairenin içinde?
        poi_within_buffer = gpd.sjoin(houses_gdf2, poi_gdf2, how='left', predicate='contains')
        poi_within_buffer = poi_within_buffer.dropna(subset=['index_right'])
        # -> Hücre başına kaç eşleşme oldu = o yarıçaptaki POI sayısı
        poi_counts = poi_within_buffer.groupby(poi_within_buffer.index).size()
        count_column = f'number_{stop_type}_within_{distance}m'
        # -> Hiç POI'si olmayan hücreler eşleşmede yok; onları 0 yap
        df[count_column] = poi_counts.reindex(houses_gdf.index).fillna(0).astype(int)
        houses_gdf2 = houses_gdf2.drop(columns=buffer_column)
        houses_gdf2 = houses_gdf2.set_geometry('geometry')
    return df


# >>> DEĞİŞTİ: aynı işi yapan hızlı sürüm
def count_poi_within_radius_fast(df, houses_gdf, poi_gdf, distances, stop_type):
    """Daire çizip kesiştirmek yerine BallTree'ye 'şu yarıçapta kaç nokta var' diye sor."""
    tree = BallTree(np.radians(poi_gdf[['lat', 'lon']].values), metric='haversine')
    cc = np.radians(df[['latitude', 'longitude']].values)
    for distance in distances:
        # -> count_only=True: noktaları döndürme, sadece SAY (çok daha hızlı)
        df[f'number_{stop_type}_within_{distance}m'] = tree.query_radius(
            cc, r=distance / 6371000.0, count_only=True).astype(int)
    return df

In [ ]:
# ==========================================================================
# ADIM 8b - TUM KATEGORILERI CALISTIR
# ==========================================================================

# -> Bütün kategorileri tek sözlükte topla. Anahtarlar sonraki adımlarda sütun adı olacak.
poi_gdfs = {
    'metro': metro_gdf,        'kdgr': kindergardens_gdf,  'schl': schools_gdf,
    'univ': university_gdf,    'bus': bus_stops_gdf,       'shopping': shopping_gdf,
    'parks': parks_gdf,        'museum': museums_gdf,      'theatre': theatres_gdf,
    'coffee': coffeeshops_gdf, 'restaurant': restaurants_gdf, 'fast_food': fastfood_gdf,
    'bars': bars_gdf,          'sport': sport_gdf,         'hotel': hotels_gdf,
    'monuments': monuments_gdf,'concert_hall': concert_halls_gdf
}
poi_names = list(poi_gdfs.keys())
buffer_distances = [500, 1000, 1500, 2000, 3000]  # Distances in meters

def number_poi_within_radius(df):
    geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
    houses_gdf = gpd.GeoDataFrame(df, geometry=geometry)
    houses_gdf = houses_gdf.set_crs("EPSG:4326")
    fn = count_poi_within_radius_fast if HIZLI_SAYIM else count_poi_within_radius
    # -> 17 kategori x 5 yarıçap = 85 yeni sütun
    for label, poi_gdf in poi_gdfs.items():
        df = fn(df, houses_gdf, poi_gdf, buffer_distances, label)
    return df

import time
t = time.time()
df = number_poi_within_radius(df)
print(f"{'hızlı' if HIZLI_SAYIM else 'ORİJİNAL'} yöntem · {time.time()-t:.1f} sn · tablo {df.shape}")

In [ ]:
# ==========================================================================
# ADIM 9 - HALKALARA BOL
#   Sayimlar KUMULATIF: '1000 m icinde 12 kafe' zaten 500 m'dekileri de
#   iceriyor. Agirliklandirma icin her halkanin KENDI sayisi lazim:
#   500-1000 m arasinda kac tane?
# ==========================================================================

import pandas as pd

bands = ['500m', '1000m','1500m', '2000m', '3000m']
new_cols = {}

for poi in poi_names:
    for i, band in enumerate(bands):
        col = f'number_{poi}_within_{band}'
        if i == 0:
            # -> İlk halka zaten kendi başına (0-500 m)
            new_cols[f'{poi}_incr_{band}'] = df[col]
        else:
            # -> Diğerleri: bu yarıçaptaki toplam - bir önceki yarıçaptaki toplam
            prev_col = f'number_{poi}_within_{bands[i-1]}'
            new_cols[f'{poi}_incr_{band}'] = df[col] - df[prev_col]

# -> Sütunları tek seferde ekle. Tek tek eklemek pandas'ı çok yavaşlatıyor (fragmentation).
new_cols_df = pd.DataFrame(new_cols, index=df.index)
df = pd.concat([df, new_cols_df], axis=1)
df.shape

In [ ]:
# ==========================================================================
# ADIM 10 - 'VEYD VERMEK' - MESAFEYE GORE AGIRLIKLANDIRMA
#   SISTEMIN KALBI. Yakindaki POI cok, uzaktaki az sayilmali.
#     1) Mesafe -> yurume suresine cevrilir (80 m/dakika)
#     2) Her halkanin sayisi exp(-beta x dakika) ile carpilir
#     3) Toplam = o kategorinin skoru
#   beta buyukse hizli duser (uzaga yurunmez), kucukse yavas duser.
# ==========================================================================

# Define speed explicitly (100m/minute = 6 km/h)
METERS_PER_MINUTE = 80  # assuming average walking speed is 80 meters per minute
# -> DİKKAT: üstteki İngilizce yorum 100 m/dk diyor, kod 80 kullanıyor. Çelişki, ekibe sorulacak.

band_distances = {'500m': 500, '1000m': 1000, '1500m': 1500, '2000m': 2000, '3000m': 3000}
# -> Her halkanın DIŞ kenarının kaç dakika sürdüğü.
# -> Sorun: halkadaki HER POI bu dış kenar süresini alıyor. 10 m'deki kafe ile
# -> 499 m'deki kafe aynı puanı veriyor — raporda B5.
band_times = {band: dist / METERS_PER_MINUTE for band, dist in band_distances.items()}

###chosen based on error correlation
# -> Bu 17 sayının nereden geldiği belgelenmemiş. Tek not yukarıdaki satır — S6.
# -> metro 0.05 = yavaş düşer (metroya uzağa da yürünür)
# -> bus 0.7   = çok hızlı düşer (500 m'den sonra pratikte sıfır)
decay_weights = {
    'kdgr': 0.1, 'schl': 0.2, 'univ': 0.1, 'bus': 0.7, 'metro': 0.05,
    'shopping': 0.1, 'parks': 0.1, 'museum': 0.1, 'theatre': 0.1,
    'coffee': 0.5, 'restaurant': 0.3, 'fast_food': 0.3, 'bars': 0.2,
    'sport': 0.2, 'hotel': 0.1, 'monuments': 0.1, 'concert_hall': 0.05
}

def calculate_poi_score(df, poi, decay_beta):
    """Calculate decay-weighted accessibility score for a POI type."""
    score = 0
    for band, time_ in band_times.items():
        incr_col = f'{poi}_incr_{band}'
        if incr_col not in df.columns:
            raise ValueError(f"Column {incr_col} missing. Check input data.")
        weight = np.exp(-decay_beta * time_)   # -> üstel azalım: süre arttıkça ağırlık düşer
        score += df[incr_col] * weight         # -> o halkadaki sayı x ağırlık
    return score

for poi in poi_names:
    df[f'{poi}_score'] = calculate_poi_score(df, poi, decay_weights[poi])

# -> β'nın etkisini göster: bus'ta 1000 m'den sonrası pratikte sıfır
print("halka ağırlıkları:")
for p in ['metro', 'schl', 'coffee', 'bus']:
    w = {b: round(float(np.exp(-decay_weights[p]*t)), 4) for b, t in band_times.items()}
    print(f"  {p:<8} β={decay_weights[p]:<5} {w}")

In [ ]:
# ==========================================================================
# OLU KOD - _normalized
#   Bu hucre orijinalde var, calisiyor - ama urettigi sutunlar HICBIR YERDE
#   kullanilmiyor. PCA _score uzerinden calisiyor. Silmedim.
# ==========================================================================

###Normalize scores
# -> min-max normalizasyon: en küçük 0, en büyük 1 olacak şekilde sıkıştır.
# -> Sorun: tek bir aykırı hücre paydayı belirliyor, şehrin %99'u dibe yığılıyor.
for poi in poi_names:
    df[f'{poi}_normalized']=(df[f'{poi}_score']-df[f'{poi}_score'].min())/(df[f'{poi}_score'].max()-df[f'{poi}_score'].min())

print("üretiliyor ama kullanılmıyor ->", [c for c in df.columns if c.endswith('_normalized')][:3], "...")

In [ ]:
# ==========================================================================
# ADIM 11 - PCA - INDEKSLERIN DOGDUGU YER
#   Ekibin bahsettigi principal component analysis.
#   Tehsil icin uc skor var: bagca, mekteb, universitet. PCA bu ucunun
#   ORTAK YONUNU bulur - ucunun de ayni anda yuksek/dusuk oldugu tek eksen.
#   O eksen uzerindeki konum = education_index_pca.
#   variance explained = uc skorun ne kadarini tek sayiyla anlatabildigimiz.
#   %84 demek: bilginin %84'u korunuyor, %16'si atiliyor.
#   NOT: PCA ciktisinin ISARETI KEYFIDIR. Burada hepsi pozitif cikti (40
#   rastgele alt orneklemde test ettim, donmedi) ama kod bunu hic kontrol
#   etmiyor. Ters donerse indeks sessizce tersine doner. v2'de sabitlendi.
# ==========================================================================

# -> Hangi kategoriler hangi indekse giriyor
education_cols=['kdgr_score', 'schl_score', 'univ_score']
transport_cols=['metro_score', 'bus_score']
tourism_cols=['museum_score','theatre_score', 'hotel_score','monuments_score','concert_hall_score']
food_cols=['coffee_score', 'restaurant_score', 'fast_food_score']  #exclude bars
recreation_cols=['parks_score','sport_score']
# -> DİKKAT: leisure, food ve recreation üyelerini TEKRAR kullanıyor -> indeksler birbiriyle korele
leisure_cols=['museum_score','shopping_score','parks_score','theatre_score', 'concert_hall_score',
              'coffee_score','restaurant_score','fast_food_score', 'bars_score']

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# >>> DEĞİŞTİ: orijinalde altı ayrı hücrede aynı 12 satır kopyalanmıştı; döngüye aldım.
# -> Hesaplama birebir aynı, sadece tekrar yok.
for name, cols in [('education', education_cols), ('transport', transport_cols),
                   ('tourism', tourism_cols), ('food', food_cols),
                   ('recreation', recreation_cols), ('leisure', leisure_cols)]:
    # -> 1) Standartlaştır: her skoru "ortalamadan kaç standart sapma uzakta" haline getir.
    # -> Olmazsa 3800 kafe ile 25 metro aynı terazide tartılamaz.
    scaler = StandardScaler()
    scaled_scores = scaler.fit_transform(df[cols])
    # -> 2) Birinci ana bileşeni çıkar = ortak yön
    pca = PCA(n_components=1)
    df[f'{name}_index_pca'] = pca.fit_transform(scaled_scores)
    # -> 3) Yükler: her kategorinin ortak yöne katkısı. Hepsi pozitifse indeks doğru yönde.
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    print(f"{name:<11} variance explained: {pca.explained_variance_ratio_[0]:.2%}   "
          + "  ".join(f"{c.replace('_score','')}={l[0]:+.2f}" for c, l in zip(cols, loadings)))

In [ ]:
# ==========================================================================
# ADIM 12 - INDEKSLER BIRBIRINDEN NE KADAR FARKLI?
#   1'e yakin = neredeyse ayni seyi olcuyorlar. Hepsinin %84'u tek faktor.
# ==========================================================================

idx = ['education_index_pca','leisure_index_pca','food_index_pca',
       'transport_index_pca','tourism_index_pca','recreation_index_pca']
# -> 1'e yakın = neredeyse aynı şeyi ölçüyorlar. Raporda B4: hepsinin %84'ü tek faktör.
df[idx].corr().round(2)

In [ ]:
# ==========================================================================
# ADIM 13 - DISA AKTARIM
# ==========================================================================

# -> Platforma/geliştiricilere giden sütunlar: kimlik + 5 mesafe + 6 indeks
poi_scores=df[['h3_index','distance_to_nearest_kindergarten', 'distance_to_nearest_schl',
               'distance_to_nearest_metro','distance_to_nearest_shopping_mall',
               'distance_to_nearest_parks','education_index_pca', 'leisure_index_pca',
               'food_index_pca', 'transport_index_pca', 'tourism_index_pca','recreation_index_pca']]
# >>> DEĞİŞTİ  (ORİJİNAL: '.../Receb_data/calculated poi scores/poi_pca.csv')
poi_scores.to_csv(str(OUT / "poi_pca.csv"), index=False)
print("yazıldı:", str(OUT / "poi_pca.csv"), poi_scores.shape)
poi_scores.tail(3)

In [ ]:
# ==========================================================================
# ADIM 14 - 10 UZERINDEN PUAN  ***  KIRIK OLAN YER  ***
#   pd.qcut(x, 10) = 'veriyi 10 esit parcaya bol'. Ama hucrelerin %22-52'si
#   ayni taban degerinde (etrafinda hic POI yok). Esit degerler ayni kutuya
#   girmek zorunda, duplicates='drop' da fazla kutulari SESSIZCE siliyor.
#   Sonuc: 10 kutu istiyorsun, 5 tane aliyorsun.
#   Calistir ve asagidaki ciktiya bak.
# ==========================================================================

def to_decile(s: pd.Series, suffix: str = "_decile"):
    """Convert a 1-D Series to integer deciles (1–10) and return (decile_series, cut_points)."""
    # -> labels=False -> 0..9 arası sayı döndür; +1 ile 1..10 yapılıyor
    # -> duplicates="drop" -> ASIL SORUN BURADA. Eşit kesim noktalarını sessizce siliyor.
    dec = pd.qcut(s, 10, labels=False, duplicates="drop") + 1
    # -> Kesim noktaları: gelecekte yeni veriyi yeniden fit etmeden puanlamak için.
    # -> İyi bir fikir ama diske hiç yazılmıyor, notebook kapanınca kayboluyor.
    cuts = s.quantile(np.linspace(0, 1, 11)).to_numpy()
    return dec.astype("Int8").rename(s.name + suffix), cuts

pca_cols = [c for c in poi_scores.columns if c.endswith("_pca")]
decile_edges = {}

poi_scores = poi_scores.copy()
for col in pca_cols:
    dec_col, edges = to_decile(poi_scores[col])
    poi_scores[dec_col.name] = dec_col
    decile_edges[col] = edges

print("BEKLENEN: her indeks 1..10 arası puan verir")
print("GERÇEK  :")
for col in pca_cols:
    d = poi_scores[col + "_decile"]
    print(f"  {col.replace('_index_pca',''):<11} kaç bant={d.nunique():2d}   "
          f"en yüksek puan={int(d.max()):2d}/10   "
          f"tabanda sıkışan hücre={(poi_scores[col]==poi_scores[col].min()).mean()*100:5.1f}%")